[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/47_bpe.ipynb)

# 🟡 Medium: Byte-Pair Encoding (train and apply)

*Inference & Decoding*
Implement **byte-pair encoding** — both halves: learn the merge list from a
corpus, and apply it to a new word.

### Signature
```python
def train_bpe(word_counts, num_merges):
    ...  # -> list[tuple[str, str]], the merges in the order they were learned

def apply_merges(word, merges):
    ...  # -> list[str], the word's symbols after replaying every merge
```

`word_counts` is `{"low": 5, "lower": 2, ...}`. A word starts as its list of
characters. `apply_merges` must be defined too — the judge tests both.

### The algorithm
1. Count every adjacent symbol pair across the corpus, weighted by word count.
2. Take the most frequent pair. **Ties break by taking the lexicographically
   smallest pair**, so the output is deterministic.
3. Record it and merge every occurrence, scanning each word left to right.
4. Repeat `num_merges` times, or stop early if no pair occurs more than once.

`apply_merges` replays the recorded merges **in learned order** — the order is
the algorithm; applying them in a different order gives different tokens.

### Rules
- Pure Python — no `jnp` needed, and no `tokenizers`/`sentencepiece`
- Deterministic: same input, same merge list, every time
- Left-to-right, non-overlapping merges (in `aaa`, merging `(a,a)` gives `aa a`)

### Why subword tokenization exists
Word-level vocabularies cannot represent anything they did not see in training —
every new name, typo or compound becomes `<UNK>`, and the information is gone.
Character-level has no `<UNK>` problem but makes sequences ~5x longer, and
attention is quadratic in length.

BPE splits the difference: frequent words stay single tokens, rare ones
decompose into reusable pieces, and because the base vocabulary is all
bytes/characters, **nothing is ever unrepresentable**. That is the property that
matters — an open vocabulary at a fixed model size.

### What it costs
Tokenization is a frozen, corpus-dependent preprocessing step, and its seams
leak into model behaviour: arithmetic is bad partly because numbers tokenize
inconsistently, character-level tasks ("how many r's in strawberry") are hard
because the model never sees characters, and languages under-represented in the
training corpus get far more tokens per word — a direct cost and context-length
penalty for those users.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def train_bpe(word_counts, num_merges):
    """Learn BPE merges from a corpus.

    Args:
        word_counts: {word: frequency}
        num_merges:  maximum number of merges to learn

    Returns:
        list[tuple[str, str]] — merges in the order learned.
    """
    pass  # Replace this


def apply_merges(word, merges):
    """Apply a learned merge list to a word.

    Args:
        word:   the string to tokenize
        merges: list[tuple[str, str]] from train_bpe

    Returns:
        list[str] of symbols.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

merges = train_bpe(corpus, num_merges=10)
for i, m in enumerate(merges, 1):
    print(f"{i:>2}. {m[0]!r} + {m[1]!r} -> {m[0] + m[1]!r}")

print()
for w in ("lowest", "newer", "wildest"):
    print(f"{w:>8} -> {apply_merges(w, merges)}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("bpe")

# hint("bpe")      # stuck? nudge without the answer
# solution("bpe")  # spoiler: the reference implementation